# Análisis FBG: 4 Sensores + Media Polarizaciones (02/02/2026, 16:00-17:30)

Este notebook analiza **4 sensores FBG** (8 canales con 2 polarizaciones cada uno).
Calcula la **media de las polarizaciones** P y S para cada sensor y permite **definir plateaus manualmente**.

## 📋 Configuración:
- **8 RTDs**: 0-3 interiores (1=profundo, 2=medio, 4=superior, 3=superficial), 4-5 vacíos, 6-7 cápsula fibra
- **4 FBGs**: Cada uno con polarización P y S (4 × 2 = 8 canales)
- **Análisis**: Media de polarizaciones vs temperatura
- **Plateaus**: Definición manual opcional
- **Acceso a datos**: XRootD remoto vía uproot (no requiere montar /eos/ localmente)

## 1️⃣ Importar Librerías

In [1]:
# Reiniciar kernel si hay problemas con imports
import sys
print(f"Python ejecutable: {sys.executable}")
print(f"Python versión: {sys.version}")

# Verificar que scipy está disponible
try:
    import scipy
    print("✅ scipy disponible")
except ImportError:
    print("❌ scipy no encontrado - instalando...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scipy"])
    print("✅ scipy instalado - reinicia el kernel")

Python ejecutable: /Users/vicky/Desktop/rtd-calibration-ana/.venv/bin/python
Python versión: 3.9.6 (default, Dec  2 2025, 07:27:58) 
[Clang 17.0.0 (clang-1700.6.3.2)]
✅ scipy disponible


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import datetime
from scipy import stats
import uproot

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


## 2️⃣ Cargar Datos del Archivo ROOT

### 🔍 Listar Archivos Disponibles (Opcional)

Si no estás seguro de qué archivos ROOT existen en el directorio, ejecuta esta celda para listarlos.

In [ ]:
# ============================================================================
# CELDA OPCIONAL: Listar archivos .root disponibles en el directorio
# ============================================================================
# Ejecuta esta celda solo si necesitas ver qué archivos existen

try:
    import uproot
    
    # Intentar listar archivos en el directorio
    # Nota: XRootD no soporta listado de directorios directamente con uproot
    # Esta es una alternativa que intenta abrir archivos comunes
    
    base_path = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/"
    print("🔍 Intentando detectar archivos ROOT disponibles...")
    print("=" * 70)
    
    # Lista de fechas comunes a probar (últimos 2 años)
    test_dates = [
        "20250515", "20250228", "20250202",
        "20260202", "20260515", "20260101",
        "20240515", "20240228", "20240101"
    ]
    
    found_files = []
    for date in test_dates:
        filepath = f"{base_path}{date}.root"
        try:
            with uproot.open(filepath) as f:
                # Si llega aquí, el archivo existe
                tree = f["resampled_data"]
                n_entries = tree.num_entries
                found_files.append((date, n_entries))
                print(f"✓ {date}.root  ({n_entries:,} entradas)")
        except (FileNotFoundError, KeyError):
            pass  # Archivo no existe
        except Exception as e:
            pass  # Otro error, ignorar
    
    print("=" * 70)
    if found_files:
        print(f"\n✅ Archivos encontrados: {len(found_files)}")
        print("\n💡 Para usar uno de estos archivos:")
        print("   filepath = \"/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/YYYYMMDD.root\"")
    else:
        print("\n⚠️  No se encontraron archivos con fechas comunes.")
        print("\n💡 Verifica la fecha exacta en CERNBox web:")
        print("   https://cernbox.cern.ch/files/")
        
except Exception as e:
    print(f"❌ Error al intentar listar archivos: {e}")
    print("\n💡 Para verificar archivos manualmente:")
    print("   1. Accede a: https://cernbox.cern.ch/")
    print("   2. Navega a tu directorio FBGdata/ROOTFiles/pressure_setup/")
    print("   3. Anota la fecha exacta del archivo (formato: YYYYMMDD.root)")

In [ ]:
# 🔧 CONFIGURA AQUÍ LA RUTA DE TU ARCHIVO
# ⚠️ IMPORTANTE: uproot puede acceder directamente a /eos/ vía XRootD (protocolo remoto de CERN)
# NO necesitas montar CERNBox localmente - uproot descarga los datos automáticamente

# Archivos confirmados que existen:
# - 20250515.root (mayo 2025) ✓
# - 20250228.root (febrero 2025) ✓

# ⚠️ El archivo 20260202.root NO EXISTE en el servidor
# Opciones:
# A) Usar archivo existente para probar el notebook:
filepath = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250515.root"  # ⬅️ ARCHIVO QUE SÍ EXISTE

# B) Si necesitas febrero 2025 (no 2026):
# filepath = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250228.root"

# C) Si el experimento fue en otra fecha, verifica en CERNBox web:
# https://cernbox.cern.ch/files/

print(f"📂 Intentando acceder vía XRootD (remoto): {filepath}")
print("   uproot descargará automáticamente los datos desde CERN...")

# Cargar datos vía XRootD (acceso remoto)
try:
    with uproot.open(filepath) as file:
        tree = file["resampled_data"]
        data = tree.arrays(library="np")
    
    times_raw = data["t"]
    temp_raw = data["temp"]
    wav_raw = data["wav"]
    
    print(f"✅ Datos cargados correctamente: {len(times_raw)} puntos")
    print(f"   Temp shape: {temp_raw.shape}, Wav shape: {wav_raw.shape}")
    
    # Mostrar rango temporal disponible
    times_all = np.array([datetime.datetime.utcfromtimestamp(float(t)) for t in times_raw])
    print(f"\n📅 Rango temporal de los datos:")
    print(f"   Inicio: {times_all[0].strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   Fin:    {times_all[-1].strftime('%Y-%m-%d %H:%M:%S')}")
    
except FileNotFoundError:
    print(f"❌ ERROR: Archivo no encontrado en el servidor CERN.")
    print(f"\n💡 Archivos confirmados que existen:")
    print(f"   ✓ 20250515.root (mayo 2025)")
    print(f"   ✓ 20250228.root (febrero 2025)")
    print(f"\n🔍 Para verificar otros archivos:")
    print(f"   1. Accede a CERNBox web: https://cernbox.cern.ch/")
    print(f"   2. Navega a: /eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/")
    print(f"   3. Verifica la fecha exacta del archivo que necesitas")
    raise
except Exception as e:
    print(f"❌ ERROR al acceder al archivo: {e}")
    print(f"\n💡 Posibles causas:")
    print(f"   - Sin conexión a internet (XRootD requiere acceso remoto)")
    print(f"   - Sin permisos de acceso al directorio en CERN")
    print(f"   - uproot no configurado correctamente")
    raise

📂 Intentando acceder vía XRootD (remoto): /eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20260202.root
   uproot descargará automáticamente los datos desde CERN...
❌ ERROR: Archivo no encontrado en el servidor CERN.

💡 Verifica:
   1. La fecha es correcta (¿2025, 2026 o 2027?)
   2. El archivo existe en CERNBox:
      /eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/
   3. Formato esperado: YYYYMMDD.root (ejemplo: 20250515.root)

🔍 Archivos disponibles en ese directorio:
   - 20250515.root (mayo 2025)
   - 20250228.root (febrero 2025)
   - Otros... (verifica en CERNBox web)


FileNotFoundError: [Errno 2] No such file or directory: '/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20260202.root'

## 3️⃣ Filtrar Rango Temporal (16:00-17:30)

In [ ]:
# Convertir timestamps
times = np.array([datetime.datetime.utcfromtimestamp(float(t)) for t in times_raw])

print(f"📅 Rango COMPLETO de datos disponible:")
print(f"   Inicio: {times[0].strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Fin:    {times[-1].strftime('%Y-%m-%d %H:%M:%S')}")

# ============================================================================
# ⬅️ AJUSTA AQUÍ EL RANGO TEMPORAL QUE QUIERES ANALIZAR
# ============================================================================
# Opción A: Usar TODO el rango disponible
use_full_range = False  # Cambiar a True para analizar todos los datos

if use_full_range:
    t_inicio = times[0]
    t_fin = times[-1]
else:
    # Opción B: Definir rango específico manualmente
    # ⚠️ AJUSTA ESTOS VALORES según el rango que viste arriba
    fecha = datetime.date(2025, 5, 15)  # ⬅️ CAMBIAR según tu archivo
    t_inicio = datetime.datetime.combine(fecha, datetime.time(16, 0))
    t_fin = datetime.datetime.combine(fecha, datetime.time(17, 30))

# Filtrar por rango temporal
mask_time = (times >= t_inicio) & (times <= t_fin)
times_filtered = times[mask_time]
temp_filtered = temp_raw[mask_time]
wav_filtered = wav_raw[mask_time]

print(f"\n✅ Rango seleccionado para análisis:")
print(f"   {t_inicio.strftime('%Y-%m-%d %H:%M')} - {t_fin.strftime('%H:%M')}")
print(f"   Puntos en el rango: {len(times_filtered):,}")

if len(times_filtered) == 0:
    print(f"\n⚠️ ADVERTENCIA: No hay datos en el rango especificado!")
    print(f"\n💡 Soluciones:")
    print(f"   1. Cambia 'use_full_range = True' para usar todos los datos")
    print(f"   2. O ajusta 't_inicio' y 't_fin' según el rango disponible (ver arriba)")

## 4️⃣ Análisis de Temperatura (8 RTDs)

In [ ]:
# Configuración RTDs
rtd_sensors = {
    0: "RTD-0 (Interior-Profundo)",
    1: "RTD-1 (Interior-Medio)",
    2: "RTD-2 (Interior)",
    3: "RTD-3 (Interior-Superficial)",
    4: "RTD-4 (Vacío)",
    5: "RTD-5 (Vacío)",
    6: "RTD-6 (Cápsula Fibra Ref)",
    7: "RTD-7 (Cápsula Fibra Ref)"
}

# Plot
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for idx, (sensor_idx, sensor_name) in enumerate(rtd_sensors.items()):
    temp_sensor = temp_filtered[:, sensor_idx]
    mask_valid = temp_sensor > 0
    times_valid = times_filtered[mask_valid]
    temp_valid = temp_sensor[mask_valid]
    
    if len(temp_valid) > 0:
        axes[idx].plot(times_valid, temp_valid, 'b-', linewidth=0.8, alpha=0.7)
        axes[idx].set_title(f"{sensor_name}\nμ={np.mean(temp_valid):.2f}K, σ={np.std(temp_valid):.3f}K", fontsize=9)
        axes[idx].set_xlabel('Tiempo', fontsize=8)
        axes[idx].set_ylabel('Temp (K)', fontsize=8)
        axes[idx].grid(True, alpha=0.3)
        axes[idx].tick_params(labelsize=7)
        for label in axes[idx].get_xticklabels():
            label.set_rotation(45)
            label.set_ha('right')

plt.tight_layout()
plt.suptitle('Perfiles de Temperatura (8 RTDs)', fontsize=13, y=1.00)
plt.show()

print("\n📊 Estadísticas:")
for sensor_idx, sensor_name in rtd_sensors.items():
    temp_sensor = temp_filtered[:, sensor_idx]
    mask_valid = temp_sensor > 0
    temp_valid = temp_sensor[mask_valid]
    if len(temp_valid) > 0:
        print(f"{sensor_name:30s}: μ={np.mean(temp_valid):6.2f}K  σ={np.std(temp_valid):6.3f}K")

## 5️⃣ Procesamiento FBG: Filtrado y Media de Polarizaciones

### 5.1 Configuración de 4 Sensores FBG

In [ ]:
# 4 sensores × 2 polarizaciones = 8 canales
fbg_config = [
    {"pol": 0, "sensor": 1, "label": "FBG-1-P"},
    {"pol": 1, "sensor": 1, "label": "FBG-1-S"},
    {"pol": 0, "sensor": 2, "label": "FBG-2-P"},
    {"pol": 1, "sensor": 2, "label": "FBG-2-S"},
    {"pol": 0, "sensor": 3, "label": "FBG-3-P"},
    {"pol": 1, "sensor": 3, "label": "FBG-3-S"},
    {"pol": 0, "sensor": 4, "label": "FBG-4-P"},
    {"pol": 1, "sensor": 4, "label": "FBG-4-S"},
]

print(f"✅ Configurados {len(fbg_config)} canales FBG (4 sensores × 2 polarizaciones)")

### 5.2 Filtrado Wavelength (μ±3σ)

In [ ]:
def filter_wavelength_data(wav_data, timestamps, sigma_threshold=3.0):
    """Filtra wavelength usando método μ±3σ"""
    mask_positive = wav_data > 0
    wav_positive = wav_data[mask_positive]
    
    mean_wav = np.mean(wav_positive)
    std_wav = np.std(wav_positive)
    threshold_min = mean_wav - sigma_threshold * std_wav
    threshold_max = mean_wav + sigma_threshold * std_wav
    
    mask_filtered = (wav_data > 0) & (wav_data >= threshold_min) & (wav_data <= threshold_max)
    
    return {
        "timestamps": timestamps[mask_filtered],
        "wavelength": wav_data[mask_filtered],
        "mean": mean_wav,
        "std": std_wav,
        "n_valid": np.sum(mask_filtered)
    }

# Procesar todos los canales
fbg_filtered_data = {}
for cfg in fbg_config:
    wav_data = wav_filtered[:, cfg["pol"], cfg["sensor"]]
    fbg_filtered_data[cfg["label"]] = filter_wavelength_data(wav_data, times_filtered)

print("✅ Filtrado completado")
print(f"\n{'Sensor':<12} {'Mean (nm)':<12} {'Std (nm)':<12} {'N válidos'}")
print("="*60)
for label, data in fbg_filtered_data.items():
    print(f"{label:<12} {data['mean']:<12.6f} {data['std']:<12.6f} {data['n_valid']}")

### 5.3 Calcular Media de Polarizaciones

In [ ]:
# Calcular media P+S para cada sensor
fbg_mean_data = {}

for sensor_num in [1, 2, 3, 4]:
    label_p = f"FBG-{sensor_num}-P"
    label_s = f"FBG-{sensor_num}-S"
    label_mean = f"FBG-{sensor_num}-Mean"
    
    data_p = fbg_filtered_data[label_p]
    data_s = fbg_filtered_data[label_s]
    
    times_p = data_p["timestamps"]
    times_s = data_s["timestamps"]
    wav_p = data_p["wavelength"]
    wav_s = data_s["wavelength"]
    
    # Interpolar S a tiempos de P
    times_p_sec = np.array([t.timestamp() for t in times_p])
    times_s_sec = np.array([t.timestamp() for t in times_s])
    wav_s_interp = np.interp(times_p_sec, times_s_sec, wav_s)
    
    # Media
    wav_mean = (wav_p + wav_s_interp) / 2.0
    
    fbg_mean_data[label_mean] = {
        "timestamps": times_p,
        "wavelength": wav_mean,
        "mean": np.mean(wav_mean),
        "std": np.std(wav_mean),
        "n_valid": len(wav_mean),
        "sensor_num": sensor_num
    }

print("✅ Medias calculadas (P+S)/2")
print(f"\n{'Sensor':<15} {'Mean (nm)':<15} {'Std (nm)':<15} {'N puntos'}")
print("="*65)
for label, data in fbg_mean_data.items():
    print(f"{label:<15} {data['mean']:<15.6f} {data['std']:<15.6f} {data['n_valid']}")

### 5.4 Visualizar Wavelengths (Media de Polarizaciones)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, (label, data) in enumerate(fbg_mean_data.items()):
    times_fbg = data["timestamps"]
    wav_fbg = data["wavelength"]
    sensor_num = data["sensor_num"]
    
    axes[idx].plot(times_fbg, wav_fbg, 'b-', linewidth=0.8, alpha=0.7, label='Media P+S')
    axes[idx].axhline(data['mean'], color='g', linestyle='--', linewidth=1.5, alpha=0.7, label='μ')
    
    axes[idx].set_title(f"FBG-{sensor_num} (Media)\nμ={data['mean']:.6f} nm, σ={data['std']:.6f} nm", fontsize=11)
    axes[idx].set_xlabel('Tiempo', fontsize=10)
    axes[idx].set_ylabel('Wavelength (nm)', fontsize=10)
    axes[idx].grid(True, alpha=0.3)
    axes[idx].legend(fontsize=9)
    axes[idx].tick_params(labelsize=9)
    
    for tick_label in axes[idx].get_xticklabels():
        tick_label.set_rotation(45)
        tick_label.set_ha('right')

plt.tight_layout()
plt.suptitle('Wavelength FBGs (Media de Polarizaciones)', fontsize=14, y=1.00)
plt.show()

## 6️⃣ Correlación Temperatura - Wavelength

### 6.1 Seleccionar RTD de Referencia

In [ ]:
# Usar RTD-6 (cápsula fibra) como referencia
rtd_ref_idx = 6
rtd_ref_name = "RTD-6 (Cápsula)"

temp_ref = temp_filtered[:, rtd_ref_idx]
mask_temp_valid = temp_ref > 0
times_temp_ref = times_filtered[mask_temp_valid]
temp_ref_valid = temp_ref[mask_temp_valid]

print(f"✅ Referencia: {rtd_ref_name}")
print(f"   Rango: [{np.min(temp_ref_valid):.2f}, {np.max(temp_ref_valid):.2f}] K")

### 6.2 Definición Manual de Plateaus (Opcional)

**✏️ Edita aquí para definir tus intervalos temporales**

In [ ]:
# ============================================================================
# DEFINICIÓN MANUAL DE PLATEAUS
# ============================================================================
# Formato: [("Nombre", "HH:MM", "HH:MM"), ...]

manual_plateaus = [
    # Ejemplo - DESCOMENTAR Y AJUSTAR:
    # ("Plateau 1", "16:10", "16:25"),
    # ("Plateau 2", "16:30", "16:45"),
    # ("Plateau 3", "17:00", "17:15"),
]

# Activar/desactivar plateaus manuales
use_manual_plateaus = False  # ⬅️ Cambiar a True para activar

plateau_masks = None

if use_manual_plateaus and len(manual_plateaus) > 0:
    print("✅ Usando plateaus manuales:")
    plateau_masks = []
    for name, t0_str, tfin_str in manual_plateaus:
        t0 = datetime.datetime.combine(fecha, datetime.datetime.strptime(t0_str, "%H:%M").time())
        tfin = datetime.datetime.combine(fecha, datetime.datetime.strptime(tfin_str, "%H:%M").time())
        plateau_masks.append((name, t0, tfin))
        print(f"   - {name}: {t0_str} - {tfin_str}")
else:
    print("ℹ️ No se usan plateaus - análisis sobre todo el rango temporal")

### 6.3 Calcular Correlaciones (Media Polarizaciones)

In [ ]:
def correlate_fbg_temperature(fbg_times, fbg_wav, temp_times, temp_values):
    """Interpola temperatura y calcula correlación"""
    fbg_seconds = np.array([t.timestamp() for t in fbg_times])
    temp_seconds = np.array([t.timestamp() for t in temp_times])
    temp_interp = np.interp(fbg_seconds, temp_seconds, temp_values)
    
    slope, intercept, r_value, p_value, std_err = stats.linregress(temp_interp, fbg_wav)
    
    return {
        "temp_interp": temp_interp,
        "slope": slope,
        "intercept": intercept,
        "r_squared": r_value**2,
        "p_value": p_value
    }

# Calcular correlaciones
correlations_mean = {}

for label, data in fbg_mean_data.items():
    fbg_times = data["timestamps"]
    fbg_wav = data["wavelength"]
    
    # Aplicar plateaus si están definidos
    if plateau_masks is not None:
        times_plateau = []
        wav_plateau = []
        for pname, t0, tfin in plateau_masks:
            mask = (fbg_times >= t0) & (fbg_times <= tfin)
            times_plateau.extend(fbg_times[mask])
            wav_plateau.extend(fbg_wav[mask])
        fbg_times = np.array(times_plateau)
        fbg_wav = np.array(wav_plateau)
    
    if len(fbg_times) > 0:
        corr = correlate_fbg_temperature(fbg_times, fbg_wav, times_temp_ref, temp_ref_valid)
        correlations_mean[label] = corr

print("✅ Correlaciones calculadas")
print(f"\n{'Sensor':<15} {'Sensibilidad (pm/K)':<22} {'R²':<12} {'p-value'}")
print("="*75)
for label in fbg_mean_data.keys():
    if label in correlations_mean:
        corr = correlations_mean[label]
        sensitivity_pm = corr['slope'] * 1000  # nm -> pm
        print(f"{label:<15} {sensitivity_pm:>10.2f} pm/K         {corr['r_squared']:<12.6f} {corr['p_value']:<.2e}")

### 6.4 Visualizar Correlaciones

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, (label, data) in enumerate(fbg_mean_data.items()):
    if label not in correlations_mean:
        continue
    
    corr = correlations_mean[label]
    sensor_num = data["sensor_num"]
    temp_interp = corr["temp_interp"]
    wav = data["wavelength"]
    
    # Línea de ajuste
    temp_range = np.array([temp_interp.min(), temp_interp.max()])
    wav_fit = corr['slope'] * temp_range + corr['intercept']
    
    axes[idx].scatter(temp_interp, wav, alpha=0.3, s=10, color='blue')
    axes[idx].plot(temp_range, wav_fit, 'r-', linewidth=2, label='Ajuste lineal')
    
    sensitivity_pm = corr['slope'] * 1000
    axes[idx].set_title(f"FBG-{sensor_num} (Media)\nSens: {sensitivity_pm:.2f} pm/K, R²={corr['r_squared']:.4f}", fontsize=11)
    axes[idx].set_xlabel(f'{rtd_ref_name} (K)', fontsize=10)
    axes[idx].set_ylabel('Wavelength (nm)', fontsize=10)
    axes[idx].grid(True, alpha=0.3)
    axes[idx].legend(fontsize=9)

plt.tight_layout()
plt.suptitle(f'Correlación Wavelength vs {rtd_ref_name}', fontsize=14, y=1.00)
plt.show()

## 7️⃣ Evolución Temporal (Ejemplo)

In [ ]:
# FBG-1 vs RTD-6
fbg_example = "FBG-1-Mean"
data_example = fbg_mean_data[fbg_example]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

ax1.plot(times_temp_ref, temp_ref_valid, 'b-', linewidth=1.2, label=rtd_ref_name)
ax1.set_ylabel('Temperatura (K)', fontsize=11, color='b')
ax1.tick_params(axis='y', labelcolor='b')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper left')
ax1.set_title(f'Evolución Temporal: {fbg_example} vs {rtd_ref_name}', fontsize=13)

ax2.plot(data_example["timestamps"], data_example["wavelength"], 'r-', linewidth=0.8, label=fbg_example)
ax2.set_ylabel('Wavelength (nm)', fontsize=11, color='r')
ax2.set_xlabel('Tiempo', fontsize=11)
ax2.tick_params(axis='y', labelcolor='r')
ax2.grid(True, alpha=0.3)
ax2.legend(loc='upper left')

for label in ax2.get_xticklabels():
    label.set_rotation(45)
    label.set_ha('right')

plt.tight_layout()
plt.show()

## 8️⃣ Resumen Final

In [ ]:
print("\n" + "="*80)
print("RESUMEN COMPLETO DEL ANÁLISIS")
print("="*80)
print(f"Archivo: {filepath}")
print(f"Rango: {t_inicio.strftime('%Y-%m-%d %H:%M')} - {t_fin.strftime('%H:%M')}")
print(f"Temperatura referencia: {rtd_ref_name}")
print(f"Plateaus manuales: {'SÍ' if use_manual_plateaus else 'NO'}")
print("\nSENSIBILIDADES TÉRMICAS (Media P+S):")
print("="*80)
print(f"{'Sensor':<15} {'Sensibilidad':<18} {'R²':<10} {'Calidad'}")
print("="*80)

all_sensitivities = []
for label, data in fbg_mean_data.items():
    if label not in correlations_mean:
        continue
    
    sensor_num = data["sensor_num"]
    corr = correlations_mean[label]
    sensitivity_pm = corr['slope'] * 1000
    all_sensitivities.append(sensitivity_pm)
    
    r2 = corr['r_squared']
    if r2 > 0.99:
        quality = "Excelente ✓✓✓"
    elif r2 > 0.95:
        quality = "Muy bueno ✓✓"
    elif r2 > 0.90:
        quality = "Bueno ✓"
    else:
        quality = "Regular ⚠"
    
    print(f"FBG-{sensor_num}        {sensitivity_pm:>10.2f} pm/K     {r2:<10.6f} {quality}")

print("="*80)
print(f"\nESTADÍSTICAS GLOBALES:")
print(f"  Media: {np.mean(all_sensitivities):.2f} pm/K")
print(f"  Std: {np.std(all_sensitivities):.3f} pm/K")
print(f"  Rango: [{np.min(all_sensitivities):.2f}, {np.max(all_sensitivities):.2f}] pm/K")
print("="*80)

## 📝 Guía de Uso

### Pasos básicos:
1. **Celda 2**: Cambiar ruta del archivo ROOT
2. **Celda 3**: Ajustar rango horario si es necesario
3. **Celda 6.2**: Definir plateaus manualmente (opcional)
4. **Ejecutar todo**: Run All

### Configuración:
- **RTD referencia**: Celda 6.1 → `rtd_ref_idx` (6 o 7 para cápsula fibra)
- **Plateaus**: Celda 6.2 → editar `manual_plateaus` y `use_manual_plateaus = True`
- **Rango temporal**: Celda 3 → `t_inicio`, `t_fin`

### Output:
- Gráficos de temperatura (8 RTDs)
- Wavelengths medias (4 FBGs)
- Correlaciones Temp-Wavelength
- Sensibilidades térmicas en pm/K
- Valores R² para calidad

### Ejemplo de plateaus:
```python
manual_plateaus = [
    ("Enfriamiento inicial", "16:00", "16:20"),
    ("Estable", "16:25", "16:50"),
    ("Final", "17:00", "17:25"),
]
use_manual_plateaus = True
```